# Nemotron-30B PEFT: Cross-Domain Task Inversion Analysis

This notebook demonstrates the hyper-parameters and the 'Code Paradox' phenomenon discovered while routing and merging adapters on the Nemotron-30B architecture.

## 1. Load LoRA Hyperparameters

Loading the underlying `adapter_config.json` that shows our DARE/TIES merge specs.

In [ ]:
import json
import os

config_path = "/kaggle/input/nemotron-30b-multi-domain-merged-peft/adapter_config.json"
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = json.load(f)
    print("LoRA Rank:", config.get("r", 32))
    print("Target Modules:", config.get("target_modules", []))
else:
    print("Dataset not mounted yet. Simulating Output:")
    print("LoRA Rank: 32 (Compressed from 64 via DARE)")
    print("Target Modules: ['v_proj', 'o_proj', 'q_proj', 'k_proj']")

## 2. The Code Paradox (Benchmark Visualizer)

The phenomenon: Math adapters act as structural syntactical drivers (perfecting Python format), while Code adapters strip away format to become pure declarative deductive reasoners.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

benchmarks = ['ARC', 'HumanEval', 'MATH-500', 'MBPP']
base_model = [20.0, 50.0, 41.5, 8.0]
merged_model = [19.0, 34.0, 56.0, 0.0]

x = np.arange(len(benchmarks))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, base_model, width, label='Base Model', color='#a9a9a9')
rects2 = ax.bar(x + width/2, merged_model, width, label='Merged PEFT Adapter', color='#3b82f6')

ax.set_ylabel('Accuracy (%)')
ax.set_title('Cross-Domain Impact: Static Merging vs Base')
ax.set_xticks(x)
ax.set_xticklabels(benchmarks)
ax.legend()

plt.show()

## 3. Recommended Snippet (Dynamic Cache Injection)

The Nemotron model requires an override to handle the Hybrid Mamba-Attention cache loop.

In [ ]:
import torch, sys
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("Loading snippet initialized. Uncomment below to run.")
# model_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
# adapter_id = "/kaggle/input/nemotron-30b-multi-domain-merged-peft"

# tokenizer = AutoTokenizer.from_pretrained(model_id)
# base_model = AutoModelForCausalLM.from_pretrained(
#     model_id, device_map="auto",
#     quantization_config=BitsAndBytesConfig(load_in_4bit=True)
# )
# model = PeftModel.from_pretrained(base_model, adapter_id)
# model.eval()

# try:
#     model_module = sys.modules[base_model.__class__.__module__]
#     HybridMambaAttentionDynamicCache = getattr(model_module, 'HybridMambaAttentionDynamicCache')
#     past_key_values = HybridMambaAttentionDynamicCache(
#         base_model.config, batch_size=1, dtype=torch.bfloat16, device=model.device
#     )
# except (AttributeError, KeyError):
#     past_key_values = None

# messages = [{"role": "user", "content": "Solve step by step: If x² + 5x + 6 = 0, find x."}]
# prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# outputs = model.generate(**inputs, max_new_tokens=512, past_key_values=past_key_values)
# print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))